# Atividade — População 2010 x 2022

Processamento da tabela do Censo Demográfico 2022, seguindo o mesmo padrão de exploração, limpeza e organização usado na Aula 3.

In [5]:


import pandas as pd
from pathlib import Path


In [3]:

path = Path(r"H:\python-1\3 etp\atv 4\CD2022_Populacao_2010_Compatibilizada_20231222.xlsx")
print("Arquivo:", path)
print("Arquivo existe:", path.exists())


Arquivo: H:\python-1\3 etp\atv 4\CD2022_Populacao_2010_Compatibilizada_20231222.xlsx
Arquivo existe: True


## 1. Ler todas as abas do arquivo

In [ ]:

todas = pd.read_excel(path, sheet_name=None, header=None, engine="openpyxl")
print(todas.keys())


dict_keys(['Municípios'])


## 2. Ler a aba de municípios

In [7]:

bruto = pd.read_excel(
    path,
    sheet_name="Municípios",
    header=None,
    engine="openpyxl"
)

bruto


,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Censo Demográfico 2022: População e Domicílios...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
4,NaN,RO,11,00023,Ariquemes,90353,90353,96833
...,...,...,...,...,...,...,...,...
5572,NaN,DF,53,00108,Brasília,2570160,2572159,2817381
5573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5574,NaN,Nota: Para o cálculo das taxas de crescimento ...,NaN,NaN,NaN,NaN,NaN,NaN
5575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Limpar e organizar os dados

In [8]:

dados = bruto.iloc[3:, 1:8].copy()


dados.columns = [
    "UF",
    "COD_UF",
    "COD_MUNIC",
    "MUNICIPIO",
    "POP_2010_SINOPSE",
    "POP_2010_COMPATIBILIZADA",
    "POP_2022"
]

for col in [
    "COD_UF",
    "COD_MUNIC",
    "POP_2010_SINOPSE",
    "POP_2010_COMPATIBILIZADA",
    "POP_2022"
]:
    dados[col] = pd.to_numeric(dados[col], errors="coerce")

dados = dados.dropna(
    subset=[
        "UF",
        "MUNICIPIO",
        "POP_2010_COMPATIBILIZADA",
        "POP_2022"
    ]
).copy()


dados["CRESCIMENTO_2022_2010"] = (
    dados["POP_2022"] - dados["POP_2010_COMPATIBILIZADA"]
)

dados.head()


,UF,COD_UF,COD_MUNIC,MUNICIPIO,POP_2010_SINOPSE,POP_2010_COMPATIBILIZADA,POP_2022,CRESCIMENTO_2022_2010
3,RO,11.0,15.0,Alta Floresta D'Oeste,24392.0,24392.0,21494.0,-2898.0
4,RO,11.0,23.0,Ariquemes,90353.0,90353.0,96833.0,6480.0
5,RO,11.0,31.0,Cabixi,6313.0,6313.0,5351.0,-962.0
6,RO,11.0,49.0,Cacoal,78574.0,78574.0,86887.0,8313.0
7,RO,11.0,56.0,Cerejeiras,17029.0,17029.0,15890.0,-1139.0


## 4. Conferir os dados

In [9]:

print("Quantidade de municípios:", len(dados))
print()
dados.info()


Quantidade de municípios: 5570

<class 'pandas.DataFrame'>
RangeIndex: 5570 entries, 3 to 5572
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   UF                        5570 non-null   str    
 1   COD_UF                    5570 non-null   float64
 2   COD_MUNIC                 5570 non-null   float64
 3   MUNICIPIO                 5570 non-null   str    
 4   POP_2010_SINOPSE          5570 non-null   float64
 5   POP_2010_COMPATIBILIZADA  5570 non-null   float64
 6   POP_2022                  5570 non-null   float64
 7   CRESCIMENTO_2022_2010     5570 non-null   float64
dtypes: float64(6), str(2)
memory usage: 348.3 KB


In [10]:

dados[
    ["POP_2010_COMPATIBILIZADA", "POP_2022", "CRESCIMENTO_2022_2010"]
].describe().round(2)


,POP_2010_COMPATIBILIZADA,POP_2022,CRESCIMENTO_2022_2010
count,5570.00,5570.00,5570.00
mean,34247.00,36459.74,2212.74
std,203037.82,206518.73,11898.43
min,805.00,833.00,-257978.00
25%,5236.50,5228.00,-349.75
50%,10896.50,11065.00,125.00
75%,23413.00,24427.25,1280.75
max,11253503.00,11451999.00,261675.00


## 5. População agregada por estado

In [11]:

estado = (
    dados.groupby(["UF", "COD_UF"], as_index=False)
    .agg(
        POP_2010=("POP_2010_COMPATIBILIZADA", "sum"),
        POP_2022=("POP_2022", "sum")
    )
)


estado["CRESCIMENTO_2022_2010"] = (
    estado["POP_2022"] - estado["POP_2010"]
)


estado = estado.sort_values(
    "CRESCIMENTO_2022_2010",
    ascending=False
)

estado


,UF,COD_UF,POP_2010,POP_2022,CRESCIMENTO_2022_2010
25,SP,35.0,41262199.0,44411238.0,3149039.0
23,SC,42.0,6248436.0,7610361.0,1361925.0
8,GO,52.0,6001789.0,7056495.0,1054706.0
17,PR,41.0,10444526.0,11444380.0,999854.0
10,MG,31.0,19597330.0,20539989.0,942659.0
12,MT,51.0,3035122.0,3658649.0,623527.0
13,PA,15.0,7581051.0,8120131.0,539080.0
2,AM,13.0,3483985.0,3941613.0,457628.0
5,CE,23.0,8451644.0,8794957.0,343313.0
7,ES,32.0,3514952.0,3833712.0,318760.0


## 6. Estado com maior e menor crescimento

In [12]:
maior = estado.iloc[0]
menor = estado.iloc[-1]

print("Maior crescimento:")
print(
    f"{maior['UF']} — "
    f"{int(maior['CRESCIMENTO_2022_2010']):,} habitantes"
)

print()
print("Menor crescimento:")
print(
    f"{menor['UF']} — "
    f"{int(menor['CRESCIMENTO_2022_2010']):,} habitantes"
)


Maior crescimento:
SP — 3,149,039 habitantes

Menor crescimento:
AL — 6,796 habitantes


## 7. Salvar população agregada por estado

In [13]:
estado.to_csv(
    "populacao_agregada_por_estado.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo: populacao_agregada_por_estado.csv")


Arquivo salvo: populacao_agregada_por_estado.csv


## 8. População por município

In [14]:

municipio = dados[
    [
        "UF",
        "COD_UF",
        "COD_MUNIC",
        "MUNICIPIO",
        "POP_2010_COMPATIBILIZADA",
        "POP_2022",
        "CRESCIMENTO_2022_2010"
    ]
].copy()


municipio = municipio.sort_values(
    "CRESCIMENTO_2022_2010",
    ascending=False
)

municipio.head(20)


,UF,COD_UF,COD_MUNIC,MUNICIPIO,POP_2010_COMPATIBILIZADA,POP_2022,CRESCIMENTO_2022_2010
114,AM,13.0,2603.0,Manaus,1802014.0,2063689.0,261675.0
5572,DF,53.0,108.0,Brasília,2572159.0,2817381.0,245222.0
3832,SP,35.0,50308.0,São Paulo,11253503.0,11451999.0,198496.0
3851,SP,35.0,52205.0,Sorocaba,586816.0,723682.0,136866.0
5420,GO,52.0,8707.0,Goiânia,1301912.0,1437366.0,135454.0
141,RR,14.0,100.0,Boa Vista,284313.0,413486.0,129173.0
4401,SC,42.0,5407.0,Florianópolis,421240.0,537211.0,115971.0
243,PA,15.0,5536.0,Parauapebas,153908.0,267836.0,113928.0
5125,MS,50.0,2704.0,Campo Grande,786774.0,898100.0,111326.0
1340,PB,25.0,7507.0,João Pessoa,723515.0,833932.0,110417.0


## 9. Municípios que mais cresceram

In [15]:
print("10 municípios com maior crescimento populacional:")

municipio.head(10)[
    [
        "UF",
        "MUNICIPIO",
        "POP_2010_COMPATIBILIZADA",
        "POP_2022",
        "CRESCIMENTO_2022_2010"
    ]
]


10 municípios com maior crescimento populacional:


,UF,MUNICIPIO,POP_2010_COMPATIBILIZADA,POP_2022,CRESCIMENTO_2022_2010
114,AM,Manaus,1802014.0,2063689.0,261675.0
5572,DF,Brasília,2572159.0,2817381.0,245222.0
3832,SP,São Paulo,11253503.0,11451999.0,198496.0
3851,SP,Sorocaba,586816.0,723682.0,136866.0
5420,GO,Goiânia,1301912.0,1437366.0,135454.0
141,RR,Boa Vista,284313.0,413486.0,129173.0
4401,SC,Florianópolis,421240.0,537211.0,115971.0
243,PA,Parauapebas,153908.0,267836.0,113928.0
5125,MS,Campo Grande,786774.0,898100.0,111326.0
1340,PB,João Pessoa,723515.0,833932.0,110417.0


## 10. Municípios que mais perderam população

In [16]:

municipio_menor = municipio.sort_values(
    "CRESCIMENTO_2022_2010",
    ascending=True
)

print("10 municípios com maior redução populacional:")

municipio_menor.head(10)[
    [
        "UF",
        "MUNICIPIO",
        "POP_2010_COMPATIBILIZADA",
        "POP_2022",
        "CRESCIMENTO_2022_2010"
    ]
]


10 municípios com maior redução populacional:


,UF,MUNICIPIO,POP_2010_COMPATIBILIZADA,POP_2022,CRESCIMENTO_2022_2010
2165,BA,Salvador,2675656.0,2417678.0,-257978.0
3245,RJ,Rio de Janeiro,6320446.0,6211223.0,-109223.0
3250,RJ,São Gonçalo,999728.0,896744.0,-102984.0
172,PA,Belém,1393399.0,1303403.0,-89996.0
4934,RS,Porto Alegre,1409351.0,1332845.0,-76506.0
2312,MG,Belo Horizonte,2375609.0,2315560.0,-60049.0
1166,RN,Natal,803739.0,751300.0,-52439.0
1599,PE,Recife,1537704.0,1488920.0,-48784.0
3202,RJ,Duque de Caxias,855088.0,808161.0,-46927.0
952,CE,Fortaleza,2459712.0,2428708.0,-31004.0


## 11. Salvar população por município

In [17]:
municipio.to_csv(
    "populacao_por_municipio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo: populacao_por_municipio.csv")


Arquivo salvo: populacao_por_municipio.csv
